# TMDB 영화 데이터셋 전처리 실습

## 학습 목표
1. 두 개의 CSV 파일을 병합하여 하나의 통합 데이터프레임을 만듭니다.
2. 문자열로 저장된 JSON 컬럼을 파이썬 리스트/딕셔너리로 변환합니다.
3. 파생 변수(장르, 감독, 배우, 인지도 점수)를 직접 계산합니다.
4. 결측값·이상값을 처리하고 최종 데이터셋을 CSV로 저장합니다.

## 데이터 구조
```
tmdb_5000_movies.csv   ─┐
                        ├─ title 기준 병합 ─→ df (통합 데이터프레임)
tmdb_5000_credits.csv  ─┘
```

## 전처리 파이프라인
```
[1] 데이터 로드 & 병합
    ↓
[2] 보조 함수 정의 (JSON 파싱, 이름 추출, 감독 추출)
    ↓
[3] 파생 변수 생성 (장르/감독/배우/날짜/제작사 규모)
    ↓
[4] 결측값·이상값 제거
    ↓
[5] 감독·배우 인지도 점수 계산
    ↓
[6] 마케팅 점수 생성 (Min-Max Scaling)
    ↓
[7] 최종 컬럼 선택 & CSV 저장
```


## Step 1: 라이브러리 임포트 & 파일 경로 설정


In [3]:
# ✅ [제공 코드] 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import re                       # 정규표현식 (텍스트 패턴 찾기)
from ast import literal_eval    # 문자열 → 파이썬 리스트/딕셔너리 안전 변환

# 파일 경로 정의
movies_path = "tmdb_5000_movies.csv"
credits_path = "tmdb_5000_credits.csv"
output_path = "movie_box_office_prediction_tmdb_like.csv"

print('✅ 라이브러리 로드 완료')


✅ 라이브러리 로드 완료


## Step 2: 원본 데이터 로드 & 병합

- `tmdb_5000_movies.csv`: 영화 기본 정보 (제목, 장르, 예산, 수익 등)
- `tmdb_5000_credits.csv`: 출연진(cast) / 제작진(crew) 정보
- 두 파일을 **`title`(영화 제목)** 컬럼 기준으로 **LEFT JOIN** 합니다.


In [4]:
# ✏️ TODO 1: 두 CSV를 로드하고 병합하세요

# (1) 각 CSV 파일을 읽어서 데이터프레임으로 저장하세요
movies  = pd.read_csv(movies_path) # ← movies_path
credits = pd.read_csv(credits_path) # ← credits_path

print(f'movies  shape: {movies.shape}')
print(f'credits shape: {credits.shape}')

# (2) movies와 credits를 'title' 컬럼 기준으로 LEFT JOIN 하세요
#     힌트: pd.DataFrame.merge(other, on='컬럼명', how='left')
df = movies.merge(credits, on='title', how='left') # ← 올바른 값을 넣으세요.

print(f'\n병합 후 shape: {df.shape}')
df.head(3)


movies  shape: (4803, 20)
credits shape: (4803, 4)

병합 후 shape: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


## Step 3: 전처리 보조 함수 정의

TMDB 데이터의 장르·배우·제작진 컬럼은 CSV에 **문자열(str) 형태**로 저장되어 있습니다.

```
# CSV에 저장된 실제 값 (문자열)
'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'

# 우리가 원하는 형태 (파이썬 리스트)
[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]
```

이를 위해 세 가지 헬퍼 함수를 완성합니다.


In [5]:
# ✏️ TODO 2: safe_literal_eval 함수를 완성하세요
def safe_literal_eval(x):
    """
    CSV에 저장된 문자열 형태의 리스트/딕셔너리를 실제 파이썬 객체로 변환합니다.
    값이 비어있거나(NaN) 변환에 실패하면 빈 리스트 []를 반환합니다.
    """
    if pd.isna(x):  # ← 올바른 값을 넣으세요.(값이 NaN이면 빈 리스트 반환)
        return []
    try:
        return literal_eval(x) # ← 올바른 값을 넣으세요.(문자열을 파이썬 객체로 변환)
    except:
        return []

# 5개 컬럼에 일괄 적용
for col in ["genres", "keywords", "production_companies", "cast", "crew"]:
    if col in df.columns:
        df[col] = df[col].apply(safe_literal_eval) # ← 올바른 값을 넣으세요.

# 변환 결과 확인
print('genres 컬럼 첫 번째 값 타입:', type(df['genres'].iloc[0]))
print('genres 첫 번째 값:', df['genres'].iloc[0][:2], '...')


genres 컬럼 첫 번째 값 타입: <class 'list'>
genres 첫 번째 값: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}] ...


In [6]:
# ✏️ TODO 3: extract_names 함수를 완성하세요
def extract_names(items, top_n=None):
    """
    [{'name': '액션'}, {'name': '스릴러'}] 형태의 리스트에서
    'name' 값만 추출하여 문자열 리스트로 반환합니다.
    top_n을 지정하면 상위 N개만 반환합니다.
    """
    if not isinstance(items, list): # ← 올바른 값을 넣으세요. (리스트가 아니면 빈 리스트 반환)
        return []
    names = [
        item.get('name', "")           # ← 올바른 값을 넣으세요. (딕셔너리에서 'name' 키 추출)
        for item in items
        if isinstance(item, dict) and item.get('name')
    ]
    return names[:top_n] if top_n else names

# 테스트
sample = [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}]
print('전체 추출:', extract_names(sample))
print('상위 2개:', extract_names(sample, top_n=2))


전체 추출: ['Action', 'Adventure', 'Fantasy']
상위 2개: ['Action', 'Adventure']


In [7]:
# ✏️ TODO 4: extract_director 함수를 완성하세요
def extract_director(crew_list):
    """
    제작진(crew) 리스트에서 job이 'Director'인 사람의 이름을 반환합니다.
    감독이 없으면 np.nan을 반환합니다.
    """
    if not isinstance(crew_list, list):
        return np.nan
    for person in crew_list:
        if isinstance(person, dict) and person.get('job')  == 'Director': # ← 올바른 값을 넣으세요.
            return person.get('name') # ← 올바른 값을 넣으세요.
    return np.nan

# 테스트
sample_crew = [
    {'job': 'Producer',  'name': 'John Smith'},
    {'job': 'Director',  'name': 'James Cameron'},
    {'job': 'Screenplay','name': 'Jane Doe'}
]
print('감독 추출 결과:', extract_director(sample_crew))


감독 추출 결과: James Cameron


## Step 4: 파생 변수 생성

앞서 만든 함수를 이용해 새로운 분석 컬럼을 생성합니다.

| 새 컬럼 | 설명 | 원본 컬럼 |
|---|---|---|
| `genre` | 첫 번째 장르 이름 | `genres` |
| `director_name` | 감독 이름 | `crew` |
| `cast_names` | 주연 배우 3명 리스트 | `cast` |
| `release_month` | 개봉 월 | `release_date` |
| `release_year` | 개봉 연도 | `release_date` |
| `production_company_size` | 참여 제작사 수 | `production_companies` |


In [8]:
# ✏️ TODO 5: 파생 변수를 생성하세요

# (1) 장르: 첫 번째 장르 1개 추출 (없으면 'Unknown')
df['genre'] = df['genres'].apply(
    lambda x: extract_names(x, 1)[0]   # ← 올바른 값을 넣으세요. (상위 1개)
    if len(extract_names(x, 1)) > 0 else 'Unknown'
)

# (2) 감독 이름
df['director_name'] = df['crew'].apply(extract_director) # ← 올바른 값을 넣으세요.

# (3) 주연 배우 3명 리스트
df['cast_names'] = df['cast'].apply(lambda x: extract_names(x, 3)) # ← 올바른 값을 넣으세요.

# (4) 수치형 컬럼 강제 변환 (오류값은 NaN 처리)
for col in ['runtime', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity']:
    df[col] = pd.to_numeric(df[col], errors='coerce') # ← 올바른 값을 넣으세요.

# (5) 개봉일 → 월/연도 분리
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month  # ← 올바른 값을 넣으세요.(월)
df['release_year']  = df['release_date'].dt.year  # ← 올바른 값을 넣으세요.(년)

# (6) 참여 제작사 수
df['production_company_size'] = df['production_companies'].apply(
    lambda x: len(extract_names(x)) # ← 올바른 값을 넣으세요.
)

print('파생 변수 생성 완료')
print(df[['title', 'genre', 'director_name', 'cast_names', 'release_year', 'production_company_size']].head(3))


파생 변수 생성 완료
                                      title      genre   director_name  \
0                                    Avatar     Action   James Cameron   
1  Pirates of the Caribbean: At World's End  Adventure  Gore Verbinski   
2                                   Spectre     Action      Sam Mendes   

                                         cast_names  release_year  \
0  [Sam Worthington, Zoe Saldana, Sigourney Weaver]        2009.0   
1     [Johnny Depp, Orlando Bloom, Keira Knightley]        2007.0   
2      [Daniel Craig, Christoph Waltz, Léa Seydoux]        2015.0   

   production_company_size  
0                        4  
1                        3  
2                        3  


## Step 5: 결측값 & 이상값 제거

**제작비(budget), 수익(revenue), 상영시간(runtime)** 이 셋 중 하나라도 0이거나 NaN인 영화는
데이터 수집 오류로 간주하여 제거합니다.


In [9]:
# ✏️ TODO 6: 비정상 데이터 행을 제거하세요
before = len(df)

df = df[
    (df['budget'].fillna(0) > 0) &   # ← 올바른 값을 넣으세요.
    (df['revenue'].fillna(0) > 0) &   # ← 올바른 값을 넣으세요.
    (df['runtime'].fillna(0) > 0)     # ← 올바른 값을 넣으세요.
].copy()

after = len(df)
print(f'제거 전: {before}행  →  제거 후: {after}행  (제거: {before - after}행)')
print(f'결측 요약:\n{df[["budget", "revenue", "runtime", "director_name"]].isnull().sum()}')


제거 전: 4809행  →  제거 후: 3232행  (제거: 1577행)
결측 요약:
budget           0
revenue          0
runtime          0
director_name    2
dtype: int64


## Step 6: 감독 & 배우 인지도(Popularity) 점수 계산

영화 자체의 `popularity`를 활용해 **감독·배우별 평균 대중성 점수**를 역산합니다.

```
감독 인지도 = 해당 감독이 연출한 모든 영화의 평균 popularity
배우 인지도 = 해당 배우가 출연한 모든 영화의 평균 popularity
```


In [11]:
# ✏️ TODO 7: 감독 인지도 점수를 계산하고 df에 합치세요
director_pop = (
    df.groupby('director_name')['popularity']           # ← 올바른 값을 넣으세요.
    .mean()
    .reset_index()
    .rename(columns={'popularity': 'director_popularity_raw'}) # ← 올바른 값을 넣으세요.
)

df = df.merge(director_pop, on='director_name', how='left') # ← 올바른 값을 넣으세요.

print('감독 인지도 샘플:')
print(director_pop.sort_values('director_popularity_raw', ascending=False).head(5))


감독 인지도 샘플:
        director_name  director_popularity_raw
781        Kyle Balda               875.581305
1341       Tim Miller               514.569956
546        James Gunn               246.651596
227   Colin Trevorrow               221.947277
246   Damien Chazelle               192.528841


In [12]:
# ✏️ TODO 8: 배우 인지도 점수를 계산하고 df에 합치세요

# (1) cast_names 리스트를 행으로 펼치기 (explode)
#     힌트: DataFrame.explode('컬럼명')
cast_exploded = df[['title', 'popularity', 'cast_names']].explode('cast_names') # ← 올바른 값을 넣으세요.
cast_exploded = cast_exploded.rename(columns={'cast_names': 'cast_name'})
cast_exploded = cast_exploded.dropna(subset=['cast_name']) # ← 올바른 값을 넣으세요.

# (2) 배우별 평균 인지도 계산
actor_pop = (
    cast_exploded.groupby('cast_name')['popularity']
    .mean()
    .reset_index()
    .rename(columns={'popularity': 'actor_popularity_raw'}) # ← 올바른 값을 넣으세요.
)
cast_exploded = cast_exploded.merge(actor_pop, on='cast_name', how='left')

# (3) 영화별 평균 배우 인지도 (상위 3명 평균)
movie_cast_pop = (
    cast_exploded.groupby('title')['actor_popularity_raw']  # ← 올바른 값을 넣으세요.
    .mean()
    .reset_index()
    .rename(columns={'actor_popularity_raw': 'cast_popularity_raw'})
)

df = df.merge(movie_cast_pop, on='title', how='left') # ← 올바른 값을 넣으세요.

print('배우 인지도 샘플:')
print(actor_pop.sort_values('actor_popularity_raw', ascending=False).head(5))


배우 인지도 샘플:
            cast_name  actor_popularity_raw
2508  Morena Baccarin            514.569956
777     Dave Bautista            481.098624
1753         Jon Hamm            446.446869
948         Ed Skrein            269.786336
3094      Scott Adsit            203.734590


## Step 7: 마케팅 점수(marketing_score) 생성

세 가지 지표를 **Min-Max 정규화(0~100점)**한 뒤 가중 합산합니다.

```
marketing_score = popularity × 50%
               + log(vote_count) × 30%
               + log(budget) × 20%
```

**Min-Max Scaling 공식**:
$$score = \frac{x - x_{min}}{x_{max} - x_{min}} \times 100$$


In [ ]:
# ✏️ TODO 9: minmax_series 함수를 완성하고 marketing_score를 계산하세요
def minmax_series(s):
    """
    시리즈를 0~100 범위로 Min-Max 정규화합니다.
    NaN은 중앙값(median)으로 채웁니다.
    """
    s     = s.fillna(s.median()) # ← 올바른 값을 넣으세요.(중간값)
    min_v = s.min()
    max_v = s.max()
    if max_v == min_v:
        return pd.Series([50.0] * len(s), index=s.index)
    return (s - min_v) / (max_v - min_v) * 100 # ← min_v, max_v, min_v

# 각 지표를 0~100으로 정규화
df['popularity_scaled']  = minmax_series(df['popularity'])                      # ← 올바른 값을 넣으세요.
df['vote_count_scaled']  = minmax_series(np.log1p(df['vote_count'].fillna(0)))
df['budget_scaled']      = minmax_series(np.log1p(df['budget'].fillna(0))) # ← 올바른 값을 넣으세요.

# 가중 합산 → marketing_score
df['marketing_score'] = (
    df['popularity_scaled']  * 0.5 +   # ← 0.5
    df['vote_count_scaled']  * 0.3 +   # ← 0.3
    df['budget_scaled']      * 0.2     # ← 0.2
)

print('marketing_score 통계:')
print(df['marketing_score'].describe().round(2))


## Step 8: 최종 컬럼 선택 & CSV 저장

분석·모델링에 필요한 컬럼만 선택하여 최종 데이터셋을 저장합니다.


In [ ]:
# ✅ [제공 코드] 최종 컬럼 정의
final_cols = [
    'title', 'genre', 'director_name', 'cast_names',
    'budget', 'revenue', 'runtime',
    'vote_average', 'vote_count', 'popularity',
    'release_month', 'release_year', 'production_company_size',
    'director_popularity_raw', 'cast_popularity_raw',
    'marketing_score'
]
final_cols_exist = [c for c in final_cols if c in df.columns]
df_final = df[final_cols_exist].copy()
print(f'최종 데이터셋: {df_final.shape[0]}행 × {df_final.shape[1]}열')
df_final.head()


In [ ]:
# ✏️ TODO 10: 최종 데이터셋을 CSV로 저장하세요

# cast_names 컬럼은 리스트 타입이므로 문자열로 변환 후 저장
df_save = df_final.copy()
df_save['cast_names'] = df_save['cast_names'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else x
)

df_save.to_csv(
    _____,          # ← output_path
    index=_____,    # ← False  (인덱스는 저장하지 않음)
    encoding=_____  # ← 'utf-8-sig'  (엑셀에서 한글 깨짐 방지)
)

print(f'✅ 저장 완료: {output_path}')
print(f'   최종 shape: {df_save.shape}')
print(f'\n컬럼 목록:')
for c in df_save.columns:
    print(f'  - {c}: {df_save[c].dtype}  (결측: {df_save[c].isnull().sum()}개)')


## Step 9: 데이터 탐색 (EDA)

저장한 최종 데이터셋을 불러와 기본 통계와 분포를 확인해보세요.


In [ ]:
# ✏️ TODO 11: 최종 CSV를 다시 불러와 아래 질문에 답하는 코드를 작성하세요
df_check = pd.read_csv(output_path) # ← output_path

# Q1. 장르별 영화 수 상위 10개
print('=== 장르별 영화 수 Top 10 ===')
print(df_check['genre'].value_counts().head(10)) # ← 올바른 값을 넣으세요.

# Q2. 수익 상위 5개 영화
print('\n=== 수익 Top 5 영화 ===')
print(df_check.nlargest(5, 'revenue')[['title', 'revenue', 'budget', 'genre']]) # ← 올바른 값을 넣으세요.

# Q3. marketing_score 상위 5개 영화
print('\n=== 마케팅 점수 Top 5 영화 ===')
print(df_check.nlargest(5, 'marketing_score')[['title', 'marketing_score', 'director_name']])


## 💡 심화 도전
1. `revenue / budget`으로 **ROI(투자수익률)** 컬럼을 추가하고 ROI 상위 10개 영화를 확인하세요.
2. `release_month`별 평균 `revenue`를 막대그래프로 시각화하세요. (어떤 달이 흥행에 유리한가?)
3. `director_popularity_raw`가 높은 감독과 낮은 감독의 평균 `revenue` 차이를 비교해보세요.
4. `vote_average` 기준 8.0 이상 영화만 추출하여 장르 분포를 파이차트로 그려보세요.
